# COVID-19 EDA — covid_19_clean_complete.csv
Lab 2 style revision practice

## 1. Setup & Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 1. Load the dataset and display the first 5 rows
df = pd.read_csv(r"/mnt/e/SXC/SEM4/SUBJECTS/AI/AI-LABS/LABS/LAB 2/COVID_19_EDA_DATASET/covid_19_clean_complete.csv")
df.head()

In [ ]:
# 2. Check the shape of the dataset (rows, columns)
print(df.shape)

In [ ]:
# 3. Check column names and data types
df.info()

## 2. Initial Inspection

In [ ]:
# 4. Statistical summary of numeric columns
df.describe()

In [ ]:
# 5. List all unique countries in the dataset
countries = df['Country/Region'].unique()
print(countries)
print(f"Total countries: {len(countries)}")

In [ ]:
# 6. Check the date range covered by the dataset
print(df['Date'].min())
print(df['Date'].max())

## 3. Cleaning

In [ ]:
# 7. Convert the Date column from string to datetime
df['Date'] = pd.to_datetime(df['Date'])
df.info()

In [ ]:
# 8. Check missing values in each column
print(df.isnull().sum())

# Note: Province/State has many missing values because most countries
# don't report at the state/province level. This is expected, not an error.

In [ ]:
# 9. Check for duplicate rows
print(df.duplicated().sum())

In [ ]:
# 10. Rename column headers to lowercase and replace spaces/slashes for easier access
df.columns = df.columns.str.lower().str.replace('/', '_')
print(df.columns)

## 4. Univariate & Aggregate Analysis

In [ ]:
# 11. Find global totals (confirmed, deaths, recovered) for the latest date
latest_date = df['date'].max()
latest_data = df[df['date'] == latest_date]

total_confirmed = latest_data['confirmed'].sum()
total_deaths = latest_data['deaths'].sum()
total_recovered = latest_data['recovered'].sum()

print(f"Total Confirmed: {total_confirmed}")
print(f"Total Deaths: {total_deaths}")
print(f"Total Recovered: {total_recovered}")

In [ ]:
# 12. Find the top 10 countries by confirmed cases (latest date)
top10 = latest_data.groupby('country_region')['confirmed'].sum().sort_values(ascending=False).head(10)
print(top10)

In [ ]:
# 13. Calculate the global death rate (deaths / confirmed)
death_rate = (total_deaths / total_confirmed) * 100
print(f"Global Death Rate: {death_rate:.2f}%")

In [ ]:
# 14. Calculate death rate for each of the top 10 countries
top10_countries = top10.index
death_rate_by_country = latest_data[latest_data['country_region'].isin(top10_countries)].groupby('country_region').apply(
    lambda x: (x['deaths'].sum() / x['confirmed'].sum()) * 100
)
print(death_rate_by_country.sort_values(ascending=False))

## 5. Time-Series Visualization

In [ ]:
# 15. Plot the global trend of confirmed, deaths, and recovered over time
global_trend = df.groupby('date')[['confirmed', 'deaths', 'recovered']].sum()

plt.figure(figsize=(10,6))
plt.plot(global_trend.index, global_trend['confirmed'], label='Confirmed')
plt.plot(global_trend.index, global_trend['deaths'], label='Deaths')
plt.plot(global_trend.index, global_trend['recovered'], label='Recovered')
plt.xlabel('Date')
plt.ylabel('Count')
plt.title('Global COVID-19 Trend Over Time')
plt.legend()
plt.show()

In [ ]:
# 16. Plot confirmed cases trend for the top 5 countries
top5_countries = top10.index[:5]

plt.figure(figsize=(10,6))
for country in top5_countries:
    country_data = df[df['country_region'] == country].groupby('date')['confirmed'].sum()
    plt.plot(country_data.index, country_data.values, label=country)

plt.xlabel('Date')
plt.ylabel('Confirmed Cases')
plt.title('Confirmed Cases Trend — Top 5 Countries')
plt.legend()
plt.show()

In [ ]:
# 17. Bar chart of top 10 countries by confirmed cases
plt.figure(figsize=(10,6))
top10.plot(kind='bar', color='steelblue')
plt.xlabel('Country')
plt.ylabel('Confirmed Cases')
plt.title('Top 10 Countries by Confirmed Cases')
plt.xticks(rotation=45)
plt.show()

## 6. Correlation & Relationships

In [ ]:
# 18. Select numeric columns and compute correlation
numeric_cols = ['confirmed', 'deaths', 'recovered', 'active']
corr = df[numeric_cols].corr()
print(corr)

In [ ]:
# 19. Visualize the correlation matrix as a heatmap
plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 7. Save Cleaned Dataset

In [ ]:
# 20. Save the cleaned DataFrame to a new CSV file
df.to_csv('cleaned_covid_19.csv', index=False)
print("DataFrame successfully saved as 'cleaned_covid_19.csv'")